## Section 1: Setup and Load

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

df = pd.read_parquet('outputs/final_dataset.parquet')
print('Shape:', df.shape)
print()
print('dtypes value counts:')
print(df.dtypes.value_counts())
print()
print('First 3 column names:', list(df.columns[:3]))
print('Last 3 column names:', list(df.columns[-3:]))
print()
print('Target distribution:')
vc = df['readmitted_30d'].value_counts()
pct = df['readmitted_30d'].value_counts(normalize=True) * 100
print(pd.DataFrame({'count': vc, 'pct': pct.round(1)}))
rate = df['readmitted_30d'].mean() * 100
print(f'Readmission rate: {rate:.1f}%')

## Section 2: Identify Column Groups

In [ ]:
non_feature_cols_spec = [
    'subject_id', 'hadm_id', 'admittime', 'dischtime',
    'next_admittime', 'days_to_next', 'readmitted_30d',
    'hospital_expire_flag', 'outcome_class'
]
non_feature_cols = [c for c in non_feature_cols_spec if c in df.columns]

clinical_cols_spec = [
    'gender', 'age_at_admission', 'admission_type', 'insurance',
    'race', 'discharge_location', 'marital_status', 'los_days',
    'came_via_ed', 'prior_admissions_count'
]
clinical_cols = [c for c in clinical_cols_spec if c in df.columns]

ccsr_cols = [c for c in df.columns
             if c not in set(non_feature_cols) and c not in set(clinical_cols)]

print('Total columns:', len(df.columns))
print('Non-feature columns found:', len(non_feature_cols))
print(' ', non_feature_cols)
print('Clinical columns found:', len(clinical_cols))
print(' ', clinical_cols)
print('CCSR columns found:', len(ccsr_cols))

## Section 3: Drop Leakage and Invalid Rows

In [ ]:
# Step 1: Drop days_to_next -- derived from future admission dates, pure leakage
if 'days_to_next' in df.columns:
    df = df.drop(columns=['days_to_next'])
    print('Dropped: days_to_next (future-derived leakage)')
else:
    print('days_to_next not found -- nothing to drop')
print('Shape after step 1:', df.shape)

# Step 2: Drop rows where discharge_location == DIED
print()
print('discharge_location value counts:')
print(df['discharge_location'].value_counts(dropna=False))

died_mask = df['discharge_location'] == 'DIED'
died_count = int(died_mask.sum())
if died_count > 0:
    df = df[~died_mask].copy()
    print()
    print('Dropped', died_count, 'rows where discharge_location == DIED')
else:
    print()
    print('No DIED rows found -- nothing to drop')
print('Shape after step 2:', df.shape)

# Step 3: Drop remaining leakage columns if present
step3_cols = ['hospital_expire_flag', 'outcome_class', 'admittime', 'dischtime', 'next_admittime']
dropped_step3 = [c for c in step3_cols if c in df.columns]
if dropped_step3:
    df = df.drop(columns=dropped_step3)
    print()
    print('Dropped columns:', dropped_step3)
else:
    print()
    print('None of the step-3 leakage columns were present in df')
print('Shape after step 3:', df.shape)

## Section 4: Strip CCSR Column Name Formatting

In [ ]:
# Identify CCSR columns before rename using original spec lists
ccsr_sample_before = [c for c in df.columns
                      if c not in set(non_feature_cols_spec)
                      and c not in set(clinical_cols_spec)]
print('5 sample CCSR column names BEFORE stripping:')
print(ccsr_sample_before[:5])

old_names = list(df.columns)
df.columns = df.columns.str.strip().str.replace("'", '', regex=False)
new_names = list(df.columns)
changed_count = sum(1 for o, n in zip(old_names, new_names) if o != n)

# Redefine column groups after rename
non_feature_cols = [c for c in non_feature_cols_spec if c in df.columns]
clinical_cols = [c for c in clinical_cols_spec if c in df.columns]
ccsr_cols = [c for c in df.columns
             if c not in set(non_feature_cols) and c not in set(clinical_cols)]

print()
print('5 sample CCSR column names AFTER stripping:')
print(ccsr_cols[:5])
print()
print('Columns renamed:', changed_count)
print('CCSR columns redefined:', len(ccsr_cols), 'total')

## Section 5: Handle Missing Values

In [ ]:
print('Missing values BEFORE filling:')
mv = df.isnull().sum()
has_missing = mv[mv > 0]
print(has_missing if len(has_missing) > 0 else 'None')
print()

# Fill categorical missing with UNKNOWN
for col in ['discharge_location', 'marital_status', 'insurance']:
    if col in df.columns:
        n = int(df[col].isnull().sum())
        df[col] = df[col].fillna('UNKNOWN')
        print(f'{col}: filled {n} missing values with UNKNOWN')

# Fill numeric missing with column median
for col in ['age_at_admission', 'los_days', 'prior_admissions_count']:
    if col in df.columns:
        n = int(df[col].isnull().sum())
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f'{col}: filled {n} missing values with median={med:.2f}')

print()
print('Missing values AFTER filling:')
mv_after = df.isnull().sum()
has_missing_after = mv_after[mv_after > 0]
print(has_missing_after if len(has_missing_after) > 0 else 'No missing values remaining')

## Section 6: Consolidate Race Categories

In [ ]:
n_unique_before = df['race'].nunique()
print('Unique race values BEFORE:', n_unique_before)
print()
print('Race value counts before consolidation:')
print(df['race'].value_counts(dropna=False))

race_upper = df['race'].fillna('UNKNOWN').str.upper()

conditions = [
    race_upper.str.contains('WHITE',                na=False),
    race_upper.str.contains('BLACK',                na=False),
    race_upper.str.contains('HISPANIC|LATINO',      na=False),
    race_upper.str.contains('ASIAN',                na=False),
    race_upper.str.contains('OTHER|UNABLE|UNKNOWN', na=False),
]
choices = ['WHITE', 'BLACK', 'HISPANIC', 'ASIAN', 'OTHER']
df['race'] = np.select(conditions, choices, default='UNKNOWN')

n_unique_after = df['race'].nunique()
print()
print('Unique race values AFTER:', n_unique_after)
print()
print('Race value counts after consolidation:')
print(df['race'].value_counts())

## Section 7: Encode Categorical Columns

In [ ]:
shape_before_enc = df.shape
print('Shape before encoding:', shape_before_enc)

cat_cols_to_encode = [
    'admission_type', 'insurance', 'discharge_location',
    'marital_status', 'race', 'gender'
]
cat_cols_present = [c for c in cat_cols_to_encode if c in df.columns]
print('Columns to encode:', cat_cols_present)

df = pd.get_dummies(df, columns=cat_cols_present, prefix=cat_cols_present, drop_first=False)

new_col_count = df.shape[1] - shape_before_enc[1]
print()
print('Shape after encoding:', df.shape)
print('New columns created:', new_col_count)

## Section 8: Scale Numeric Features

In [ ]:
numeric_to_scale = ['age_at_admission', 'los_days', 'prior_admissions_count', 'came_via_ed']
numeric_present = [c for c in numeric_to_scale if c in df.columns]

print('Stats BEFORE scaling:')
for col in numeric_present:
    print(f'  {col}: mean={df[col].mean():.4f}, std={df[col].std():.4f}')

scaler = StandardScaler()
df[numeric_present] = scaler.fit_transform(df[numeric_present].astype(float))

print()
print('Stats AFTER scaling:')
for col in numeric_present:
    print(f'  {col}: mean={df[col].mean():.4f}, std={df[col].std():.4f}')

## Section 9: Train Test Split by Patient

In [ ]:
patient_ids = df['subject_id'].unique()
train_ids, test_ids = train_test_split(
    patient_ids, test_size=0.2, random_state=42
)
train_df = df[df['subject_id'].isin(train_ids)].copy()
test_df  = df[df['subject_id'].isin(test_ids)].copy()

print('Train shape:', train_df.shape)
print('Test shape: ', test_df.shape)

train_rate = train_df['readmitted_30d'].mean() * 100
test_rate  = test_df['readmitted_30d'].mean() * 100
print()
print(f'Readmission rate in train: {train_rate:.1f}%')
print(f'Readmission rate in test:  {test_rate:.1f}%')

overlap = set(train_ids) & set(test_ids)
assert len(overlap) == 0, f'Leakage: {len(overlap)} patients in both splits'
print()
print(f'Data leakage check: {len(overlap)} patients in both train and test')

## Section 10: Save Clean Files

In [ ]:
id_and_target = ['subject_id', 'hadm_id', 'readmitted_30d']
feature_cols = [c for c in df.columns if c not in id_and_target]

X_train = train_df[feature_cols]
X_test  = test_df[feature_cols]
y_train = train_df['readmitted_30d']
y_test  = test_df['readmitted_30d']

X_train.to_parquet('outputs/X_train.parquet', index=False)
X_test.to_parquet('outputs/X_test.parquet',  index=False)
y_train.to_csv('outputs/y_train.csv', index=False)
y_test.to_csv('outputs/y_test.csv',   index=False)
df.to_parquet('outputs/final_dataset_clean.parquet', index=False)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
print('y_train:', y_train.shape)
print('y_test: ', y_test.shape)
print('final_dataset_clean:', df.shape)
print()

files_to_check = [
    'outputs/X_train.parquet',
    'outputs/X_test.parquet',
    'outputs/y_train.csv',
    'outputs/y_test.csv',
    'outputs/final_dataset_clean.parquet',
]
for fpath in files_to_check:
    assert os.path.exists(fpath), f'Missing: {fpath}'

print('Preprocessing complete. Ready for modeling.')